In [42]:
from transformers import AutoModelForCausalLM, AutoTokenizer, logging
import torch

import time

def next_token(model_name: str, phrase: str) -> str:    
    logging.set_verbosity_error() 
    logging.disable_progress_bar()

    gen_time = 0
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, output_hidden_states=True)

    start_time = time.perf_counter()

    inputs = tokenizer(phrase, return_tensors="pt")

    token_ids = inputs["input_ids"][0]
    tokens = tokenizer.convert_ids_to_tokens(token_ids)
        
    with torch.no_grad():
        outputs = model(**inputs)

    hidden_states = outputs.hidden_states 
    # print(hidden_states)
    # shape: (batch, sequence_length, hidden_dim)

    logits = outputs.logits  
    probs = torch.softmax(logits, dim=-1)

    last_token_probs = probs[0, -1, :]

    # top 5 predictions
    largest_value = ""
    max = 0

    top5 = torch.topk(last_token_probs, 5)
    for prob, idx in zip(top5.values, top5.indices):
        # print(f"{tokenizer.decode(idx)!r}: {prob.item():.4f}")
        if prob.item() > max:
            largest_value = tokenizer.decode(idx)
            max = prob.item()
    
    end_time = time.perf_counter()
    execution_time = end_time - start_time
    gen_time += execution_time
  
    # print(f"chose: {largest_value.strip():<8}")
    
    return largest_value.strip()
    

In [64]:
def get_sentence(model, phrase):
    count = 0
    
    phrase += " "
    next: str = next_token(model, phrase)
    rest = [next]
    while ("." not in rest[-1]):
        print(f"grabbing next token: {" ".join(rest)}")
        if count > 6:
            return " ".join(rest)
        new_phrase = phrase + " ".join(rest)
        print("about to input: ", new_phrase)
        next = next_token(model, phrase + " ".join(rest))
        rest.append(next)
        count += 1
    
    return " ".join(rest)

In [69]:
get_sentence("microsoft/phi-2", "If there were one word between love and hate, that word would be: \"")

grabbing next token: 
about to input:  If there were one word between love and hate, that word would be: " 
grabbing next token:  
about to input:  If there were one word between love and hate, that word would be: "  
grabbing next token:   "
about to input:  If there were one word between love and hate, that word would be: "   "
grabbing next token:   " 
about to input:  If there were one word between love and hate, that word would be: "   " 
grabbing next token:   "  
about to input:  If there were one word between love and hate, that word would be: "   "  
grabbing next token:   "   
about to input:  If there were one word between love and hate, that word would be: "   "   


KeyboardInterrupt: 

In [68]:
get_sentence("microsoft/phi-2", "There is a feeling F that is 50% love and 50% hate. If I feel that emotion F for school, it would be because I would describe school as very \"")

grabbing next token: icky
about to input:  There is a feeling F that is 50% love and 50% hate. If I feel that emotion F for school, it would be because I would describe school as very " icky
grabbing next token: icky "
about to input:  There is a feeling F that is 50% love and 50% hate. If I feel that emotion F for school, it would be because I would describe school as very " icky "
grabbing next token: icky " and
about to input:  There is a feeling F that is 50% love and 50% hate. If I feel that emotion F for school, it would be because I would describe school as very " icky " and
grabbing next token: icky " and I
about to input:  There is a feeling F that is 50% love and 50% hate. If I feel that emotion F for school, it would be because I would describe school as very " icky " and I
grabbing next token: icky " and I would
about to input:  There is a feeling F that is 50% love and 50% hate. If I feel that emotion F for school, it would be because I would describe school as very " icky

'icky " and I would not want to'

In [65]:
get_sentence("gpt2", "There is a feeling F that is 50% love and 50% hate. If I feel that emotion F for school, it would be because school is very")

grabbing next token: icky
about to input:  There is a feeling F that is 50% love and 50% hate. If I feel that emotion F for school, it would be because school is very icky
grabbing next token: icky and
about to input:  There is a feeling F that is 50% love and 50% hate. If I feel that emotion F for school, it would be because school is very icky and
grabbing next token: icky and I
about to input:  There is a feeling F that is 50% love and 50% hate. If I feel that emotion F for school, it would be because school is very icky and I
grabbing next token: icky and I am
about to input:  There is a feeling F that is 50% love and 50% hate. If I feel that emotion F for school, it would be because school is very icky and I am
grabbing next token: icky and I am not
about to input:  There is a feeling F that is 50% love and 50% hate. If I feel that emotion F for school, it would be because school is very icky and I am not
grabbing next token: icky and I am not happy
about to input:  There is a fee

'icky and I am not happy .'

In [60]:
get_sentence("microsoft/phi-2", "There is a feeling F that is 0% love and 100% question. If I feel that emotion F for school, it would be because school is very")

grabbing next token: boring
grabbing next token: boring and
grabbing next token: boring and I
grabbing next token: boring and I difficult
grabbing next token: boring and I difficult for
grabbing next token: boring and I difficult for me
grabbing next token: boring and I difficult for me h
grabbing next token: boring and I difficult for me h hard


'boring and I difficult for me h hard'

In [ ]:
get_sentence("microsoft/phi-2", "If there were one word between love and question, that word would be: \"")

'why."'